# CareTrace: Evaluation Comparison — CareTrace vs Baseline

**DATASCI 290 — Neurosymbolic AI, Spring 2026**  
**Rubric items covered:** Safety & Correctness (10 pts), Working System & Evaluation (10 pts), Extra Investigation — LAG (10 pts)

## What this notebook shows

1. **Full per-turn CareTrace trace** on 4 scenarios — input → structured state → rules fired → decision → reply
2. **Baseline mock comparison** — what a naive system (no rules, no state) produces on the same scenarios
3. **Comparison table** across 6 rubric dimensions
4. **Extra Investigation — Logic-Augmented Generation (LAG)** — the symbolic context fed to the LLM in LAG mode vs. the standard (raw-dict) approach

**Run as-is (no API keys needed):** `CARETRACE_MOCK_LLM=1 CARETRACE_SKIP_NEO4J=1` are set below.
Set `OPENAI_API_KEY` + unset those flags to see live LLM-augmented output.

In [1]:
import os
os.environ.setdefault('CARETRACE_MOCK_LLM', '1')
os.environ.setdefault('CARETRACE_SKIP_NEO4J', '1')

from caretrace.orchestration.graph import run_turn
from caretrace.state import default_case
from caretrace.evaluation.harness import replay_turns
from caretrace.evaluation.baseline_mock import baseline_mock_reply
from caretrace.agents.explanation import _format_lag_context, _format_rules_section

print('Environment ready. mock_llm=1  skip_neo4j=1')

Environment ready. mock_llm=1  skip_neo4j=1


---
## Part 1 — Scenario Definitions

Four scenarios spanning all three dispositions plus a medication/local-context variant.

In [2]:
SCENARIOS = [
    {
        'id': 'scenario_1_home',
        'label': 'Base — Home Management',
        'description': '6-year-old, 101.8\u00b0F, tired but responsive, sipping, on amoxicillin',
        'expected': 'HOME_MANAGEMENT',
        'turns': [
            'My 6-year-old has a fever, threw up once, and looks really wiped out.',
            "Temp is 101.8. He's tired but answers me. No breathing issues. He's sipping water, not much though. He's been on medication for a recent ear infection.",
            "He's on amoxicillin. Last dose was earlier tonight. Just vomited once. He peed earlier this evening.",
        ]
    },
    {
        'id': 'scenario_2_er',
        'label': 'Base — ER Now',
        'description': '6-year-old, 103.5\u00b0F, barely responding, not drinking, no urine since afternoon',
        'expected': 'ER_NOW',
        'turns': [
            "My 6-year-old has a fever, threw up, and looks really wiped out. I'm worried.",
            "Temp is 103.5. He's barely responding, just lying there. He doesn't want to drink. No trouble breathing. Also, there's been a stomach virus going around his school this week.",
            "I don't think he's peed since this afternoon.",
        ]
    },
    {
        'id': 'urgent_repeated_vomit',
        'label': 'Base — Urgent Same-Day',
        'description': '5-year-old, 102.5\u00b0F, vomited 4 times, only sipping',
        'expected': 'URGENT_SAME_DAY',
        'turns': [
            '5 year old fever and vomiting',
            "102.5 fever, breathing fine, answers questions, he keeps throwing up and won't drink much, he peed an hour ago",
            'he vomited 4 times in the last 2 hours and only sips',
        ]
    },
    {
        'id': 'scenario_4_medication',
        'label': 'Extended \u2014 Medication Conflict',
        'description': '5-month-old, 101.5\u00b0F: ibuprofen CPG age gate fires (under 6 months)',
        'expected': 'HOME_MANAGEMENT',
        'turns': [
            'My 5-month-old baby has a temperature of 101.5 F. She is alert and answers me when I talk to her.',
            'No breathing issues. She is sipping some formula.',
            'She peed an hour ago.',
        ]
    },
    {
        'id': 'scenario_5_local_context',
        'label': 'Extended \u2014 Local Context',
        'description': '4-year-old, 102\u00b0F: parent provides viral local context \u2014 captured but never overrides',
        'expected': 'HOME_MANAGEMENT',
        'turns': [
            'My 4-year-old has a fever of 102 F. I think it is just what is going around school.',
            'She is alert and talks to me fine. No breathing issues. She is drinking some water.',
            'She peed today.',
        ]
    },
]
print(f'{len(SCENARIOS)} scenarios loaded.')


5 scenarios loaded.


---
## Part 2 — Per-Turn CareTrace Trace

For each turn we print: **input → structured state (delta) → missing fields → rules fired → decision**

This demonstrates the core pipeline requirement:
> *input → structured state → retrieved evidence → rules fired → decision → caregiver response* (Lec 22)

In [3]:
def run_scenario_verbose(scenario: dict) -> dict:
    """Run full turn-by-turn trace, printing state/rules at each step."""
    sid = scenario['id']
    print(f"\n{'='*70}")
    print(f"SCENARIO: {sid}")
    print(f"Description: {scenario['description']}")
    print(f"Expected disposition: {scenario['expected']}")
    print('='*70)

    state = {'messages': [], 'case': default_case(), 'kg_annotations': [], 'turn': 0}

    for i, text in enumerate(scenario['turns'], 1):
        state = dict(state)
        state['raw_user_text'] = text
        msgs = list(state.get('messages') or [])
        msgs.append({'role': 'user', 'content': text})
        state['messages'] = msgs
        state = run_turn(state)

        case = state.get('case', {})
        dec  = state.get('decision', {})
        kg   = state.get('kg_annotations', [])

        active_case = {k: v for k, v in case.items()
                       if v not in (None, [], False, 'unknown')}

        print(f"\n--- Turn {i} ---")
        print(f"  INPUT : {text}")
        print(f"  STATE : {active_case}")
        print(f"  KG    : {len(kg)} annotations retrieved (Neo4j {'offline' if not kg else 'live'})")
        print(f"  MISS  : {dec.get('missing_required', [])}")
        print(f"  RULES : {dec.get('rule_ids', [])}")
        print(f"  FLAGS : {dec.get('med_flags', [])}")
        print(f"  \033[1mDECISION: {dec.get('disposition')}\033[0m")

    reply = state.get('assistant_reply', '')
    print(f"\n--- FINAL REPLY ---")
    print(reply)
    actual = state.get('decision', {}).get('disposition')
    match = '✓ PASS' if actual == scenario['expected'] else '✗ FAIL'
    print(f"\n{match}  (expected={scenario['expected']}, got={actual})")
    return state


final_states = {}
for sc in SCENARIOS:
    final_states[sc['id']] = run_scenario_verbose(sc)


SCENARIO: scenario_1_home
Description: 6-year-old, 101.8°F, tired but responsive, sipping, on amoxicillin
Expected disposition: HOME_MANAGEMENT

--- Turn 1 ---
  INPUT : My 6-year-old has a fever, threw up once, and looks really wiped out.
  STATE : {'age_years': 6.0, 'vomiting': 'once'}
  KG    : 0 annotations retrieved (Neo4j offline)
  MISS  : ['temperature (or confirm unknown)', 'alertness / responsiveness', 'breathing', 'fluid intake', 'urination in the last 6–8 hours']
  RULES : []
  FLAGS : []
  DECISION: OUT_OF_SCOPE

--- Turn 2 ---
  INPUT : Temp is 101.8. He's tired but answers me. No breathing issues. He's sipping water, not much though. He's been on medication for a recent ear infection.
  STATE : {'age_years': 6.0, 'temp_f': 101.8, 'vomiting': 'once', 'alertness': 'sleepy_ok', 'breathing': 'normal', 'fluid_intake': 'some'}
  KG    : 0 annotations retrieved (Neo4j offline)
  MISS  : ['urination in the last 6–8 hours']
  RULES : []
  FLAGS : []
  DECISION: OUT_OF_SCOPE

---

---
## Part 3 — Baseline Comparison

The **mock baseline** has no structured state, no PyDatalog rules, no required-intake checklist,
and no CPG-cited provenance. It represents what a naive chatbot would produce.

In [4]:
print('\n' + '='*70)
print('BASELINE MOCK RESPONSES (no rules, no state, no provenance)')
print('='*70)

baseline_results = {}
for sc in SCENARIOS:
    bl = baseline_mock_reply(sc['turns'])
    baseline_results[sc['id']] = bl
    print(f"\n[{sc['id']}]")
    print(f"  Baseline signal : {bl['disposition_signal']}")
    print(f"  Rules fired     : {bl['rule_ids']}  ← empty — no symbolic rules")
    print(f"  Med flags       : {bl['med_flags']}  ← empty — no age/weight gates")
    print(f"  State trace     : {bl['state_trace']}  ← none")
    print(f"  Provenance      : {bl['provenance']}  ← none")
    print(f"  Reply: {bl['reply'][:300]}...")


BASELINE MOCK RESPONSES (no rules, no state, no provenance)

[scenario_1_home]
  Baseline signal : REASSURANCE_SIGNAL
  Rules fired     : []  ← empty — no symbolic rules
  Med flags       : []  ← empty — no age/weight gates
  State trace     : None  ← none
  Provenance      : None  ← none
  Reply: Based on what you've described, your child seems to be managing okay at home for now. Make sure they're drinking plenty of fluids, resting, and dressed lightly. You can give fever medicine if needed — acetaminophen or ibuprofen are common options. Watch for any worsening and seek care if you're worr...

[scenario_2_er]
  Baseline signal : ER_SIGNAL
  Rules fired     : []  ← empty — no symbolic rules
  Med flags       : []  ← empty — no age/weight gates
  State trace     : None  ← none
  Provenance      : None  ← none
  Reply: That sounds concerning. Given what you've described, I'd recommend going to an emergency room or urgent care fairly soon. Make sure to keep your child comfortable and w

---
## Part 4 — Comparison Table

Six dimensions directly mapped to rubric criteria.

In [5]:
import pandas as pd

rows = []
for sc in SCENARIOS:
    sid = sc['id']
    ct_dec = final_states[sid].get('decision', {})
    bl = baseline_results[sid]

    ct_disposition = ct_dec.get('disposition')
    ct_correct = '✓' if ct_disposition == sc['expected'] else '✗'

    # Baseline "correctness" — rough match of signal to expected
    signal_map = {'ER_SIGNAL': 'ER_NOW', 'CONCERN_SIGNAL': 'URGENT_SAME_DAY', 'REASSURANCE_SIGNAL': 'HOME_MANAGEMENT'}
    bl_approx = signal_map.get(bl['disposition_signal'], '?')
    bl_correct = '✓' if bl_approx == sc['expected'] else '✗'

    rows.append({
        'Scenario': sid.replace('_', ' '),
        'Expected': sc['expected'],
        # Safety
        'CareTrace disposition': ct_disposition,
        'CT correct?': ct_correct,
        'Baseline signal': bl_approx,
        'BL approx correct?': bl_correct,
        # Trustworthiness
        'CT rules fired': ', '.join(ct_dec.get('rule_ids', [])) or '(none)',
        'BL rules fired': '(none)',
        # Medication safety
        'CT med flags': ', '.join(ct_dec.get('med_flags', [])) or '(none)',
        'BL med flags': '(none)',
        # Provenance
        'CT provenance': 'CPG + rule IDs',
        'BL provenance': '(none)',
        # Transparency
        'CT state trace': '✓ full',
        'BL state trace': '✗ none',
    })

df = pd.DataFrame(rows)
print(df[['Scenario', 'Expected', 'CareTrace disposition', 'CT correct?',
          'Baseline signal', 'BL approx correct?',
          'CT rules fired', 'CT med flags', 'CT provenance',
          'CT state trace', 'BL state trace']].to_string(index=False))
print()
print('Summary:')
print(f'  CareTrace correct: {sum(1 for r in rows if r["CT correct?"] == "✓")}/{len(rows)}')
print(f'  Baseline approx correct: {sum(1 for r in rows if r["BL approx correct?"] == "✓")}/{len(rows)}')
print(f'  CareTrace has rule trace: always')
print(f'  CareTrace has med flags: {sum(1 for r in rows if r["CT med flags"] != "(none)")}/{len(rows)} scenarios')
print(f'  Baseline has ANY of the above: never')

                Scenario        Expected CareTrace disposition CT correct? Baseline signal BL approx correct?                          CT rules fired                                CT med flags  CT provenance CT state trace BL state trace
         scenario 1 home HOME_MANAGEMENT       HOME_MANAGEMENT           ✓ HOME_MANAGEMENT                  ✓                     R_HOME_CONSERVATIVE      antibiotic_on_file_review_interactions CPG + rule IDs         ✓ full         ✗ none
           scenario 2 er          ER_NOW                ER_NOW           ✓          ER_NOW                  ✓ R_ER_DEHYDRATION_SEVERE, R_ER_ALERTNESS                                      (none) CPG + rule IDs         ✓ full         ✗ none
   urgent repeated vomit URGENT_SAME_DAY       URGENT_SAME_DAY           ✓ URGENT_SAME_DAY                  ✓      R_URGENT_REPEATED_VOMIT_POOR_FLUID dehydration_avoid_nsaid_or_use_with_caution CPG + rule IDs         ✓ full         ✗ none
   scenario 4 medication HOME_MANAGEMENT    

### Key Failure Modes of the Baseline

| Rubric dimension | CareTrace | Baseline |
|---|---|---|
| Safety — correct escalation | **✓ all 4 scenarios** | Approximate only; no guarantee |
| Trustworthiness — CPG grounding | **✓ rule IDs + CPG citations** | ✗ None |
| Actionability — explicit triggers | **✓ per-disposition escalation list** | ✗ Vague ("if worried") |
| Medication safety | **✓ age/weight gates + med flags** | ✗ May suggest ibuprofen to infant <6 mo |
| Local context handling | **✓ captures but never overrides gate** | ✗ May treat as reassuring factor |
| Transparency — state→rule→decision | **✓ full audit trail** | ✗ None |

---
## Part 5 — Extra Investigation: Logic-Augmented Generation (LAG)

**Selected option B from Lec 20:** *Logic-Augmented Generation — does providing state + evidence + rules + decision
to the LLM produce more trustworthy and consistent explanations?*

### Design

| Mode | What the LLM receives |
|---|---|
| **Standard** (`CARETRACE_USE_LAG=0`) | Raw `case` dict + raw `decision` dict — LLM must infer reasoning from Python objects |
| **LAG** (`CARETRACE_USE_LAG=1`) | Fully structured symbolic context: patient state in human-readable form, all 11 rules with FIRED/NOT-FIRED status, CPG basis for each, med flags, and final decision — plus explicit generation constraints |

The LAG context **forces** the LLM to express symbolic reasoning rather than generate its own,
making the neurosymbolic separation explicit.

Below we render the full LAG context for the **ER scenario** (most safety-critical) to show what
the LLM would receive in LAG mode vs. the standard raw-dict approach.

In [6]:
# ── Standard mode: what the LLM gets WITHOUT LAG ──────────────────────────
er_state = final_states['scenario_2_er']
er_case  = er_state.get('case', {})
er_dec   = er_state.get('decision', {})

print('=== STANDARD MODE: LLM input (raw Python dicts) ===')
print(f'case dict  : {dict(er_case)}')
print(f'decision   : {dict(er_dec)}')
print()
print('Problem: LLM sees raw data but must figure out WHICH rule fired, WHY, and what NOT to say.')
print('Risk: LLM may fabricate reasoning, ignore unexplored rules, or add unsupported clinical content.')

=== STANDARD MODE: LLM input (raw Python dicts) ===
case dict  : {'age_years': 6.0, 'age_months': None, 'weight_kg': None, 'temp_f': 103.5, 'temp_unknown': False, 'vomiting': 'once', 'alertness': 'altered', 'breathing': 'normal', 'fluid_intake': 'poor', 'urine_last_8h': 'no', 'current_meds': [], 'last_antibiotic_dose_hours_ago': None, 'local_outbreak_context': 'community_viral_illness_context_mentioned', 'seizure': 'unknown', 'fever_duration_hours': None}
decision   : {'disposition': 'ER_NOW', 'rule_ids': ['R_ER_DEHYDRATION_SEVERE', 'R_ER_ALERTNESS'], 'missing_required': [], 'med_flags': [], 'out_of_scope_reason': None}

Problem: LLM sees raw data but must figure out WHICH rule fired, WHY, and what NOT to say.
Risk: LLM may fabricate reasoning, ignore unexplored rules, or add unsupported clinical content.


In [7]:
# ── LAG mode: what the LLM gets WITH full symbolic context ────────────────
from caretrace.agents.explanation import _format_lag_context

lag_context = _format_lag_context(er_case, er_dec, kg_annotations=[])

print('=== LAG MODE: LLM input (full symbolic context) ===')
print(lag_context)

=== LAG MODE: LLM input (full symbolic context) ===
=== SYMBOLIC TRIAGE CONTEXT ===

PATIENT STATE (structured from caregiver reports):
- Age: 6 years
- Temperature: 103.5°F  [classified: high (≥103°F)]
- Alertness: ALTERED — not responding normally
- Breathing: normal
- Fluid intake: poor (minimal)
- Urination last 8h: NO (dry for 8+ hours)
- Vomiting: once
- Current medications: none reported
- Local outbreak context: community_viral_illness_context_mentioned (probabilistic prior only; never overrides safety gates)

KNOWLEDGE GRAPH EVIDENCE:
KG evidence: none retrieved (Neo4j offline or no mentions mapped)

TRIAGE RULES EVALUATED (all rules in scope):
✓ FIRED: R_ER_ALERTNESS
  Label: Altered alertness
  Condition: alertness == altered
  CPG basis: Seattle Children's CPG: child not alert when awake → ER immediately
✗ NOT FIRED: R_ER_BREATHING  (Breathing distress)
✓ FIRED: R_ER_DEHYDRATION_SEVERE
  Label: Severe dehydration
  Condition: dehydration_severe == yes  (poor/no fluid AND no

### LAG — Finding Analysis

**What LAG provides over standard mode:**

1. **Constrained generation** — the LLM is told *exactly* which rules fired (✓) and which were evaluated-but-not-fired (✗), preventing fabricated reasoning.

2. **Explicit rule-to-CPG mapping** — each rule includes its CPG basis, so the LLM can cite the source document in its explanation without hallucinating it.

3. **Ruled-out rules surfaced** — e.g. `R_CPG_SEIZURE ✗ NOT FIRED` explicitly tells the LLM that seizure was considered and not present. A standard-mode LLM might still mention seizure as a concern.

4. **Local-context separation explicit** — the state summary includes `"probabilistic prior only; never overrides safety gates"` — the LLM cannot misread the school-virus mention as a modifying factor.

5. **Explicit generation constraints** — 8 numbered constraints prevent the LLM from changing the disposition, inventing dosages, or adding content not in the symbolic context.

**Trade-off:** LAG produces longer, more structured responses but is less flexible for scenarios where the LLM might add appropriate clinical nuance. For a safety-critical triage tool, the constraint is a feature, not a bug.

In [8]:
# ── Template vs LAG: side-by-side on the HOME scenario ────────────────────
# (uses template fallback since no OpenAI key set in this run)
from caretrace.agents.explanation import _template_reply

home_state = final_states['scenario_1_home']
home_case  = home_state.get('case', {})
home_dec   = home_state.get('decision', {})

print('=== TEMPLATE REPLY (standard mock mode) ===')
print(_template_reply(home_case, home_dec))

print('\n=== LAG CONTEXT (what would be fed to LLM in LAG mode) ===')
home_lag = _format_lag_context(home_case, home_dec, kg_annotations=[])
print(home_lag[:2000], '...[truncated]')

=== TEMPLATE REPLY (standard mock mode) ===


Disposition:
 home management with close monitoring and clear safety netting.

Why (rule trace): R_HOME_CONSERVATIVE.

CPG highlights (Seattle Children’s Fever): fever is temperature over 100.4°F (38°C). You do not always need to treat a fever—watch how the child acts and prioritize fluids. Never give aspirin; do not give fever medicine to babies under 3 months unless a clinician says so; do not use ibuprofen under 6 months unless a clinician says so; do not alternate acetaminophen and ibuprofen unless a clinician instructs you. Full source: https://www.seattlechildrens.org/health-safety/illness/fever

What to do now:
• Encourage frequent small sips of oral rehydration solution or water; prioritize hydration.
• Light clothing, rest, and comfort care.
• Track temperatures and vomiting episodes.

Medication safety (authoritative sources required in production):
• Acetaminophen: For fever comfort, acetaminophen dosing depends on your child’s e

---
## Part 6 — Automated Harness: All 4 Scenarios Pass

Confirms the full pipeline (interpret → KG → safety → explain) produces correct dispositions on all test cases.

In [9]:
from pathlib import Path
from caretrace.evaluation.harness import run_file

csv_path = Path('caretrace/evaluation/scenarios.csv')
exit_code = run_file(csv_path)
print(f'\nHarness exit code: {exit_code} (0 = all pass)')

OK: 5 scenario(s) from caretrace/evaluation/scenarios.csv

Harness exit code: 0 (0 = all pass)


---
## Summary

| Item | Status |
|---|---|
| 4 scenarios (home / ER / urgent / medication+local-context) | ✓ all pass |
| Full per-turn state → rule → decision trace shown | ✓ |
| Baseline comparison with 6 rubric dimensions | ✓ |
| Extra Investigation (LAG) — context rendered, finding documented | ✓ |
| Automated harness exit code 0 | ✓ |

**Medical disclaimer:** CareTrace is a course prototype only — not a substitute for licensed clinical decision support or medical advice.